In [ ]:
import os, subprocess, sys, time, shutil, re

print("Stale 프로세스 정리 중...")
os.system('fuser -k 8501/tcp > /dev/null 2>&1')
os.system('pkill -f streamlit > /dev/null 2>&1')
os.system('pkill -f cloudflared > /dev/null 2>&1')

print("패키지 설치 중...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'streamlit', 'plotly', 'pandas', 'numpy',
                       'scikit-learn', 'xgboost', 'scipy', 'torch', '-q'])

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print("환경 준비 완료.")

app_lines = []

app_lines.append('''import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import os
import scipy.stats as stats
from datetime import datetime
from scipy.stats import gaussian_kde

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve,
    precision_score, recall_score, f1_score, accuracy_score,
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    from sklearn.ensemble import RandomForestClassifier
    XGB_AVAILABLE = False

st.set_page_config(
    page_title="AIS — 산업 위협 탐지 시스템",
    layout="wide",
    initial_sidebar_state="expanded",
)
''')

app_lines.append('''
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap');

html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
.stApp { background-color: #f8fafc !important; color: #1e293b !important; }

[data-testid="stSidebar"] {
    background-color: #ffffff !important;
    border-right: 1px solid #e2e8f0 !important;
}
[data-testid="stSidebar"] * { color: #475569 !important; }
[data-testid="stSidebar"] h1,
[data-testid="stSidebar"] h2,
[data-testid="stSidebar"] h3 { color: #0f172a !important; font-weight: 700 !important; }

.main .block-container { padding: 2rem 2.5rem; max-width: 100%; }

h1 { color: #0f172a !important; font-size: 1.75rem !important; font-weight: 700 !important; letter-spacing: -0.03em; }
h2 { color: #64748b !important; font-size: 1rem !important; font-weight: 400 !important; margin-bottom: 1.5rem; }
h3 { color: #0f172a !important; font-size: 1.05rem !important; font-weight: 600 !important; }

[data-testid="stMetric"] {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 16px;
    padding: 1.5rem 1.75rem;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.02);
    transition: box-shadow 0.2s;
}
[data-testid="stMetric"]:hover { box-shadow: 0 4px 12px rgba(0,0,0,0.08); }
[data-testid="stMetric"] label {
    color: #64748b !important;
    font-size: 0.7rem !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: 0.08em;
}
[data-testid="stMetricValue"] {
    color: #0f172a !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 1.85rem !important;
    font-weight: 600 !important;
}

.stButton > button {
    background: #0f172a !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 0.65rem 1.5rem !important;
    font-size: 0.85rem !important;
    font-weight: 600 !important;
    transition: all 0.2s ease;
}
.stButton > button:hover {
    background: #1e40af !important;
    transform: translateY(-1px);
    box-shadow: 0 4px 12px rgba(30,64,175,0.3) !important;
}

.sec-hdr {
    font-size: 0.7rem;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    color: #94a3b8;
    margin: 1.75rem 0 1rem 0;
    padding-bottom: 0.5rem;
    border-bottom: 2px solid #f1f5f9;
}

.badge {
    display: inline-flex;
    align-items: center;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.72rem;
    font-weight: 600;
    border-radius: 6px;
    padding: 3px 10px;
    margin: 2px;
}
.badge-danger { background: #fef2f2; color: #dc2626; border: 1px solid #fecaca; }
.badge-warn   { background: #fffbeb; color: #d97706; border: 1px solid #fde68a; }
.badge-safe   { background: #f0fdf4; color: #16a34a; border: 1px solid #bbf7d0; }

.threat-card {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-left: 4px solid #dc2626;
    border-radius: 12px;
    padding: 1rem 1.25rem;
    margin-bottom: 0.75rem;
}
.threat-card-safe {
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    border-left: 4px solid #16a34a;
    border-radius: 12px;
    padding: 1rem 1.25rem;
    margin-bottom: 0.75rem;
}

.page-header {
    background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%);
    border-radius: 20px;
    padding: 2rem 2.5rem;
    margin-bottom: 2rem;
    color: white;
}
.page-header h1 { color: #ffffff !important; font-size: 1.6rem !important; margin: 0; }
.page-header p  { color: #93c5fd; font-size: 0.85rem; margin: 0.4rem 0 0 0; }

.sidebar-sec {
    font-size: 0.68rem;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    color: #94a3b8;
    margin: 1.25rem 0 0.6rem 0;
    padding-bottom: 0.4rem;
    border-bottom: 1px solid #f1f5f9;
}

.streamlit-expanderHeader {
    background: #f8fafc !important;
    border: 1px solid #e2e8f0 !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
}
</style>
""\", unsafe_allow_html=True)
''')

app_lines.append('''
@st.cache_data
def load_dataset_ml():
    base_path = '/content/drive/MyDrive/ML/'
    file_names = ['02-14-2018.csv', '02-15-2018.csv', '02-16-2018.csv']
    frames = []
    if not os.path.exists(base_path):
        return None
    for fname in file_names:
        fpath = os.path.join(base_path, fname)
        if not os.path.exists(fpath):
            continue
        try:
            preview = pd.read_csv(fpath, nrows=5)
            preview.columns = preview.columns.str.strip()
            target_cols = ['Timestamp', 'Label', 'Dst Port']
            col_map = {
                'Flow IAT Mean':          ['Flow IAT Mean', 'Flow iat Mean'],
                'Fwd Packet Length Mean': ['Fwd Packet Length Mean', 'Fwd Pkt Len Mean'],
                'Flow Duration':          ['Flow Duration', 'Flow Dur'],
                'Flow Packets/s':         ['Flow Packets/s', 'Flow Pkts/s'],
                'Bwd Packet Length Mean': ['Bwd Packet Length Mean', 'Bwd Pkt Len Mean'],
            }
            found = {}
            for canonical, variants in col_map.items():
                for c in preview.columns:
                    if any(kw in c for kw in variants):
                        target_cols.append(c)
                        found[c] = canonical
                        break
            df = pd.read_csv(fpath, nrows=20000, usecols=list(set(target_cols)))
            df.columns = df.columns.str.strip()
            df = df.rename(columns=found)
            frames.append(df)
        except Exception:
            continue
    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined['Timestamp'] = pd.to_datetime(combined['Timestamp'], errors='coerce')
    for c in ['Flow Packets/s', 'Bwd Packet Length Mean']:
        if c in combined.columns and combined[c].dtype == object:
            combined[c] = pd.to_numeric(combined[c], errors='coerce')
    num_cols = combined.select_dtypes(include=[np.number]).columns
    combined[num_cols] = combined[num_cols].replace([np.inf, -np.inf], np.nan)
    combined = combined.dropna().sort_values('Timestamp').reset_index(drop=True)
    return combined

df_raw = load_dataset_ml()
if df_raw is None:
    st.error("데이터셋 로드 실패 — 구글 드라이브 /ML/ 경로와 CSV 파일을 확인하세요.")
    st.stop()
''')

app_lines.append('''
with st.sidebar:
    st.markdown("## AIS Control Panel")
    st.markdown('<div class=\"sidebar-sec\">운영 모드</div>', unsafe_allow_html=True)
    app_mode = st.radio(
        "페이지 선택",
        ["현장 관제 대시보드", "ML/DL 성능 검증 리포트"],
        label_visibility="collapsed",
    )
    st.markdown('<div class=\"sidebar-sec\">신경망 아키텍처</div>', unsafe_allow_html=True)
    nn_type = st.selectbox("순환 레이어 종류", ["LSTM", "GRU"])
    st.markdown('<div class=\"sidebar-sec\">탐지 임계값</div>', unsafe_allow_html=True)
    CRIT_THRES = st.slider(
        "Z-score 기준", 2.0, 6.0, 3.2, 0.1,
        help="높을수록 엄격한 탐지. 산업 현장 권장값: 3.0~3.5",
    )
    calculated_contam = max(2 * (1 - stats.norm.cdf(CRIT_THRES)), 0.005)
    st.caption("오염 비율(rho): **{:.2f}%**".format(calculated_contam * 100))
    st.markdown('<div class=\"sidebar-sec\">시스템 상태</div>', unsafe_allow_html=True)
    st.success("{} 분류기 준비됨".format("XGBoost" if XGB_AVAILABLE else "RandomForest"))
    st.info("데이터: {:,}건 로드".format(len(df_raw)))

selected_mode = nn_type
''')

app_lines.append('''
FEATURES = [
    "Flow IAT Mean", "Fwd Packet Length Mean", "Flow Duration",
    "Flow Packets/s", "Bwd Packet Length Mean",
]
available = [f for f in FEATURES if f in df_raw.columns]

df_ml    = df_raw.copy()
X_log    = np.log1p(df_ml[available])
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X_log)

def create_sequences(data, seq_length=5):
    sequences = []
    for i in range(len(data)):
        if i < seq_length:
            padding = np.repeat(data[0:1], seq_length - (i + 1), axis=0)
            seq = np.vstack([padding, data[0:i+1]])
        else:
            seq = data[i - seq_length + 1:i + 1]
        sequences.append(seq)
    return np.array(sequences)

X_seq = create_sequences(X_scaled, seq_length=5)

class RecurrentAutoencoder(nn.Module):
    def __init__(self, num_features, seq_length, mode="LSTM"):
        super().__init__()
        self.mode = mode
        RNN = nn.LSTM if mode == "LSTM" else nn.GRU
        self.encoder     = RNN(input_size=num_features, hidden_size=16, num_layers=1, batch_first=True)
        self.decoder     = RNN(input_size=16,           hidden_size=16, num_layers=1, batch_first=True)
        self.output_layer = nn.Linear(16, num_features)
        self.seq_length  = seq_length

    def forward(self, x):
        out, _      = self.encoder(x)
        last_hidden = out[:, -1, :].unsqueeze(1).repeat(1, self.seq_length, 1)
        out, _      = self.decoder(last_hidden)
        return self.output_layer(out)

@st.cache_resource
def train_deep_core(X_data, rnn_mode):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tensor_x = torch.tensor(X_data, dtype=torch.float32)
    loader   = DataLoader(TensorDataset(tensor_x), batch_size=256, shuffle=False)
    model     = RecurrentAutoencoder(X_data.shape[2], X_data.shape[1], mode=rnn_mode).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    loss_history = []
    model.train()
    for epoch in range(10):
        epoch_loss = 0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out   = model(batch)
            loss  = criterion(out, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch.size(0)
        loss_history.append(epoch_loss / len(X_data))
    model.eval()
    with torch.no_grad():
        inp    = tensor_x.to(device)
        rec    = model(inp)
        errors = torch.mean((rec[:, -1, :] - inp[:, -1, :]) ** 2, dim=1).cpu().numpy()
    return errors, loss_history

reconstruction_errors, deep_loss_curve = train_deep_core(X_seq, selected_mode)
thresh_val = np.percentile(reconstruction_errors, 100 * (1 - calculated_contam))
df_ml['anomaly_score_dl'] = reconstruction_errors
df_ml['anomaly_critical'] = reconstruction_errors > thresh_val
''')

app_lines.append('''
def raw_label_clean(label):
    s = str(label).upper()
    if 'BENIGN' in s:                              return 'Benign'
    if any(k in s for k in ['BRUTE','FTP','SSH']): return 'Brute Force'
    if 'DDOS' in s:                                return 'DDoS'
    if 'DOS'  in s:                                return 'DoS'
    if any(k in s for k in ['WEB','XSS','SQL']):   return 'Web Attack'
    return 'Other'

df_ml['Clean_Label'] = df_ml['Label'].apply(raw_label_clean)

@st.cache_resource
def train_and_evaluate_engine(X, y):
    le    = LabelEncoder()
    y_enc = le.fit_transform(y)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_enc, test_size=0.2, random_state=42, stratify=y_enc,
    )
    if XGB_AVAILABLE:
        clf = XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            eval_metric='logloss', n_jobs=-1, random_state=42,
        )
    else:
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te) if hasattr(clf, 'predict_proba') else None
    cm          = confusion_matrix(y_te, y_pred)
    report_dict = classification_report(y_te, y_pred, target_names=le.classes_, output_dict=True)
    report_df   = pd.DataFrame(report_dict).transpose().iloc[:-3, :3]
    return clf, le, cm, report_df, y_te, y_pred, y_prob

clf_model, label_encoder, conf_matrix, eval_report, y_test, y_pred, y_prob = \
    train_and_evaluate_engine(X_scaled, df_ml['Clean_Label'])

df_ml['AI_Class'] = label_encoder.inverse_transform(clf_model.predict(X_scaled))
criticals_df = df_ml[df_ml['anomaly_critical']]
safe_pct     = (1 - len(criticals_df) / len(df_ml)) * 100

CLASS_COLORS = {
    'Benign':      '#10b981',
    'Brute Force': '#f59e0b',
    'DoS':         '#ef4444',
    'DDoS':        '#8b5cf6',
    'Web Attack':  '#3b82f6',
    'Other':      '#ec4899',
}

THREAT_INFO = {
    'Brute Force': {
        'severity': 'HIGH',
        'desc': '반복적인 인증 시도로 시스템 계정 탈취를 시도하는 공격',
        'mfg':  'MES/ERP 계정 침탈 → 생산 명령 변조, 불량품 출하 지시 가능',
        'avi':  '관제 네트워크 인증 우회 → 항로 데이터 및 물류 서버 무단 접근',
        'act':  '해당 IP 즉시 차단 + 다중 인증(MFA) 적용 + 로그인 시도 횟수 제한',
    },
    'DoS': {
        'severity': 'CRITICAL',
        'desc': '단일 출처 대용량 트래픽으로 시스템 가용성을 마비시키는 공격',
        'mfg':  'PLC 제어 신호 대역폭 포화 → 실시간 공정 제어 불가, 설비 오작동',
        'avi':  '레이더 포화 공격 → 항공기 위치 추적 신호 간헐적 분실',
        'act':  '트래픽 rate limiting 즉시 적용 + 업스트림 필터링 활성화',
    },
    'DDoS': {
        'severity': 'CRITICAL',
        'desc': '분산된 다수 출처에서 동시에 시스템을 마비시키는 대규모 공격',
        'mfg':  'ICS 백본 네트워크 마비 → 제어실 전면 접근 차단, 생산 중단',
        'avi':  'ATC 서버 다운 → 데이터링크 연동 마비, 비행 안전 저하',
        'act':  'CDN/Anti-DDoS 스크러빙 센터 연동 + ISP 레벨 차단 요청',
    },
    'Web Attack': {
        'severity': 'HIGH',
        'desc': 'SQL Injection, XSS 등 웹 애플리케이션 취약점을 이용한 공격',
        'mfg':  'ERP 웹 모듈 침해 → 출하 일정 위변조, 공급망 데이터 탈취',
        'avi':  '스케줄링 시스템 교란 → 게이트 배정 오작동, 승객 혼란',
        'act':  'WAF 즉시 강화 + 취약 엔드포인트 패치 + 입력값 검증 적용',
    },
    'Other': {
        'severity': 'MEDIUM',
        'desc': 'Brute Force·DDoS·DoS·Web Attack 키워드에 매칭되지 않는 미분류 잔여 트래픽',
        'mfg':  '정상 범주에 속하지 않는 패턴 → 봇넷, 침투(Infiltration) 등 다양한 위협 혼재 가능, 추가 정밀 분류 필요',
        'avi':  '미분류 이상 신호 → 항공 관제 시스템 정밀 점검 및 수동 검토 권고',
        'act':  '해당 트래픽 격리 후 수동 분류 + 상세 로그 분석 + 필요 시 추가 위협 인텔리전스 매칭',
    },
}

PLOT_BASE = dict(
    plot_bgcolor='#ffffff', paper_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#475569', size=11),
    margin=dict(l=12, r=12, t=44, b=36),
    xaxis=dict(gridcolor='#f1f5f9', linecolor='#e2e8f0', tickfont=dict(size=9, color='#94a3b8')),
    yaxis=dict(gridcolor='#f1f5f9', linecolor='#e2e8f0', tickfont=dict(size=9, color='#94a3b8')),
    title_font=dict(size=13, color='#0f172a', family='Inter, sans-serif'),
)
''')

app_lines.append('''
if app_mode == "현장 관제 대시보드":

    now_str = datetime.now().strftime("%Y-%m-%d  %H:%M:%S")
    header_html = (
        "<div class='page-header'>"
        "<h1>AIS 산업 인프라 위협 관제 센터</h1>"
        "<p>{mode} Recurrent Autoencoder 실시간 이상 탐지 가동 중 &nbsp;|&nbsp; {ts}</p>"
        "</div>"
    ).format(mode=selected_mode, ts=now_str)
    st.markdown(header_html, unsafe_allow_html=True)

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("감시 패킷 총량",   "{:,} 건".format(len(df_ml)))
    col2.metric(
        "위협 탐지 건수",
        "{:,} 건".format(len(criticals_df)),
        delta=("위협 없음" if len(criticals_df) == 0 else "-{:.1f}% 위험도".format(safe_pct)),
    )
    col3.metric("시스템 보안 무결성", "{:.2f}%".format(safe_pct))
    col4.metric("임계 Z-score", str(CRIT_THRES),
                delta="오염률 {:.2f}%".format(calculated_contam * 100))

    st.divider()

    st.markdown('<div class="sec-hdr">스마트 방화벽 자동화 차단 (ACL Policy)</div>', unsafe_allow_html=True)
    if 'blocked_ports' not in st.session_state:
        st.session_state.blocked_ports = set()

    col_btn1, col_btn2, col_status = st.columns([1.5, 1.5, 7])
    with col_btn1:
        if st.button("AI 차단 정책 배포"):
            if len(criticals_df) > 0:
                ports = criticals_df['Dst Port'].dropna().astype(int).unique().tolist()
                st.session_state.blocked_ports.update(ports)
                st.success("{:,}개 포트 격리 완료".format(len(ports)))
    with col_btn2:
        if st.button("차단 해제"):
            st.session_state.blocked_ports.clear()
            st.info("차단 해제됨")
    with col_status:
        if st.session_state.blocked_ports:
            badges = " ".join(
                "<span class='badge badge-danger'>:{}</span>".format(p)
                for p in sorted(st.session_state.blocked_ports)
            )
            st.markdown("<b>차단 중인 포트</b> &nbsp; " + badges, unsafe_allow_html=True)
        else:
            st.caption("차단된 포트 없음 — 정상 운영 중")

    st.divider()

    col_left, col_right = st.columns([5, 5])

    with col_left:
        st.markdown('<div class="sec-hdr">탐지 위협 유형 분포</div>', unsafe_allow_html=True)

        label_counts = df_ml['AI_Class'].value_counts().reset_index()
        label_counts.columns = ['유형', '건수']
        fig_pie = go.Figure(go.Pie(
            labels=label_counts['유형'],
            values=label_counts['건수'],
            hole=0.52,
            marker_colors=[CLASS_COLORS.get(c, '#94a3b8') for c in label_counts['유형']],
            textinfo='label+percent',
            textfont_size=11,
            hovertemplate="<b>%{label}</b><br>%{value:,}건 (%{percent})<extra></extra>",
        ))
        fig_pie.update_layout(
            **PLOT_BASE,
            title="전체 트래픽 분류 현황",
            showlegend=False,
            height=280,
            annotations=[dict(
                text="<b>{:,}</b>".format(len(df_ml)),
                x=0.5, y=0.5, font_size=16, showarrow=False,
            )],
        )
        st.plotly_chart(fig_pie, use_container_width=True, config={"displayModeBar": False})

        if len(criticals_df) > 0:
            threat_counts = (
                criticals_df[criticals_df['AI_Class'] != 'Benign']['AI_Class']
                .value_counts().reset_index()
            )
            threat_counts.columns = ['유형', '건수']
            fig_bar = go.Figure(go.Bar(
                x=threat_counts['건수'],
                y=threat_counts['유형'],
                orientation='h',
                marker_color=[CLASS_COLORS.get(c, '#94a3b8') for c in threat_counts['유형']],
                marker_line_width=0,
                text=["{:,}".format(v) for v in threat_counts['건수']],
                textposition='outside',
                hovertemplate="<b>%{y}</b><br>%{x:,}건<extra></extra>",
            ))
            fig_bar.update_layout(
                **PLOT_BASE,
                title="이상 탐지 위협 유형별 건수",
                height=220,
                xaxis_title=None, yaxis_title=None,
            )
            st.plotly_chart(fig_bar, use_container_width=True, config={"displayModeBar": False})

        fig_hist = go.Figure()
        fig_hist.add_trace(go.Histogram(
            x=reconstruction_errors[~df_ml['anomaly_critical']],
            nbinsx=60, name='정상', marker_color='#10b981', opacity=0.7,
        ))
        fig_hist.add_trace(go.Histogram(
            x=reconstruction_errors[df_ml['anomaly_critical']],
            nbinsx=60, name='위협', marker_color='#ef4444', opacity=0.8,
        ))
        fig_hist.add_vline(
            x=thresh_val, line_dash='dash', line_color='#dc2626', line_width=2,
            annotation_text="임계값 {:.4f}".format(thresh_val),
            annotation_font_color='#dc2626',
        )
        fig_hist.update_layout(
            **PLOT_BASE,
            title="{} Autoencoder 복원 오차 분포".format(selected_mode),
            barmode='overlay', height=220,
            legend=dict(orientation='h', y=1.02, x=1, xanchor='right', font_size=10),
        )
        st.plotly_chart(fig_hist, use_container_width=True, config={"displayModeBar": False})

    with col_right:
        st.markdown('<div class="sec-hdr">위협별 산업 현장 영향 분석</div>', unsafe_allow_html=True)
        for cls, info in THREAT_INFO.items():
            subset = criticals_df[criticals_df['AI_Class'] == cls]
            count  = len(subset)
            if count > 0:
                card_html = (
                    "<div class='threat-card'>"
                    "<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:0.6rem;'>"
                    "<span style='font-weight:700;font-size:0.95rem;color:#0f172a;'>{cls}</span>"
                    "<div><span class='badge badge-danger'>{sev}</span>"
                    "<span class='badge badge-danger'>{cnt:,}건 탐지</span></div>"
                    "</div>"
                    "<p style='color:#64748b;font-size:0.8rem;margin:0 0 0.75rem 0;'>{desc}</p>"
                    "<div style='display:grid;grid-template-columns:1fr 1fr;gap:0.75rem;margin-bottom:0.75rem;'>"
                    "<div style='background:#fef2f2;border-radius:8px;padding:0.6rem 0.75rem;'>"
                    "<div style='font-size:0.65rem;font-weight:700;text-transform:uppercase;color:#94a3b8;margin-bottom:0.3rem;'>제조 현장</div>"
                    "<div style='font-size:0.78rem;color:#374151;line-height:1.4;'>{mfg}</div>"
                    "</div>"
                    "<div style='background:#fef2f2;border-radius:8px;padding:0.6rem 0.75rem;'>"
                    "<div style='font-size:0.65rem;font-weight:700;text-transform:uppercase;color:#94a3b8;margin-bottom:0.3rem;'>항공 관제</div>"
                    "<div style='font-size:0.78rem;color:#374151;line-height:1.4;'>{avi}</div>"
                    "</div>"
                    "</div>"
                    "<div style='background:#fffbeb;border-radius:8px;padding:0.5rem 0.75rem;border-left:3px solid #f59e0b;'>"
                    "<div style='font-size:0.65rem;font-weight:700;text-transform:uppercase;color:#94a3b8;margin-bottom:0.2rem;'>권고 조치</div>"
                    "<div style='font-size:0.78rem;color:#374151;'>{act}</div>"
                    "</div>"
                    "</div>"
                ).format(
                    cls=cls, sev=info['severity'], cnt=count,
                    desc=info['desc'], mfg=info['mfg'], avi=info['avi'], act=info['act'],
                )
                st.markdown(card_html, unsafe_allow_html=True)
            else:
                safe_html = (
                    "<div class='threat-card-safe'>"
                    "<span style='font-weight:600;font-size:0.88rem;color:#166534;'>"
                    "{cls} — 정상 범위"
                    "</span>"
                    "<p style='color:#15803d;font-size:0.78rem;margin:0.3rem 0 0 0;'>활성 징후 없음</p>"
                    "</div>"
                ).format(cls=cls)
                st.markdown(safe_html, unsafe_allow_html=True)

    st.divider()

    st.markdown('<div class="sec-hdr">피처별 실시간 트래픽 궤적</div>', unsafe_allow_html=True)
    chart_df = (df_ml.sample(n=3000, random_state=42) if len(df_ml) > 3000 else df_ml).sort_values('Timestamp')
    rows = [st.columns(3), st.columns(2)]
    for idx, feat in enumerate(available):
        slot = rows[0][idx] if idx < 3 else rows[1][idx - 3]
        fig  = go.Figure()
        for cls in chart_df['AI_Class'].unique():
            sub = chart_df[chart_df['AI_Class'] == cls]
            fig.add_trace(go.Scattergl(
                x=sub['Timestamp'], y=sub[feat],
                mode='markers', name=cls,
                marker=dict(color=CLASS_COLORS.get(cls, '#94a3b8'), size=3.5, opacity=0.55),
                hovertemplate="<b>{}</b><br>{}:  %{{y:.2f}}<extra></extra>".format(cls, feat),
            ))
        fig.update_layout(**PLOT_BASE, title=feat, showlegend=False, height=210)
        with slot:
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})
''')

app_lines.append('''
else:
    clf_name = "XGBoost" if XGB_AVAILABLE else "RandomForest"
    header_html2 = (
        "<div class='page-header'>"
        "<h1>ML/DL 모델 성능 검증 리포트</h1>"
        "<p>{mode} Autoencoder + {clf} 분류기 정량 평가</p>"
        "</div>"
    ).format(mode=selected_mode, clf=clf_name)
    st.markdown(header_html2, unsafe_allow_html=True)

    st.info(
        "이 리포트는 단순 점수 기반 탐지를 넘어 시계열 컨텍스트 의존성을 추출한 "
        "딥러닝 파이프라인의 학술적 성능 지표를 제공합니다. "
        "모든 지표는 20% Hold-out Test Set 기준입니다.",
        icon=None,
    )

    tab1, tab2, tab3, tab4 = st.tabs([
        "학습 수렴 분석",
        "혼동 행렬",
        "분류 성능 지표",
        "고급 평가 지표",
    ])

    with tab1:
        col_l1, col_l2 = st.columns(2)

        with col_l1:
            st.markdown("#### Autoencoder 학습 수렴 곡선")
            fig_loss = go.Figure()
            fig_loss.add_trace(go.Scatter(
                x=list(range(1, len(deep_loss_curve) + 1)),
                y=deep_loss_curve,
                mode='lines+markers',
                name='Training MSE Loss',
                line=dict(color='#6366f1', width=2.5),
                marker=dict(size=7, color='#6366f1'),
                fill='tozeroy',
                fillcolor='rgba(99,102,241,0.08)',
            ))
            min_ep = int(np.argmin(deep_loss_curve)) + 1
            fig_loss.add_annotation(
                x=min_ep, y=min(deep_loss_curve),
                text="최솟값 Epoch {}".format(min_ep),
                showarrow=True, arrowhead=2,
                font=dict(color='#6366f1', size=10),
                arrowcolor='#6366f1',
            )
            fig_loss.update_layout(
                **PLOT_BASE,
                title="{} Autoencoder 에포크별 MSE 손실".format(selected_mode),
                xaxis_title="Epoch", yaxis_title="MSE Loss",
                height=300,
            )
            st.plotly_chart(fig_loss, use_container_width=True, config={"displayModeBar": False})

            c1, c2, c3 = st.columns(3)
            c1.metric("초기 Loss",  "{:.5f}".format(deep_loss_curve[0]))
            c2.metric("최종 Loss",  "{:.5f}".format(deep_loss_curve[-1]))
            c3.metric("개선율", "{:.1f}%".format(
                (1 - deep_loss_curve[-1] / max(deep_loss_curve[0], 1e-9)) * 100))

        with col_l2:
            st.markdown("#### 복원 오차 KDE 분포 (정상 vs 위협)")
            fig_kde = go.Figure()
            normal_errors = reconstruction_errors[~df_ml['anomaly_critical']]
            threat_errors  = reconstruction_errors[df_ml['anomaly_critical']]

            def hex_to_rgba_str(hex_str, opacity):
                h = hex_str.lstrip('#')
                rgb = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
                return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{opacity})"

            for errs, label, color in [
                (normal_errors, '정상 트래픽', '#10b981'),
                (threat_errors,  '이상 탐지',  '#ef4444'),
            ]:
                if len(errs) > 1:
                    kde     = gaussian_kde(errs, bw_method=0.3)
                    x_range = np.linspace(errs.min(), errs.max(), 200)
                    fig_kde.add_trace(go.Scatter(
                        x=x_range, y=kde(x_range),
                        mode='lines', name=label,
                        line=dict(color=color, width=2.5),
                        fill='tozeroy',
                        fillcolor=hex_to_rgba_str(color, 0.1),
                    ))
            fig_kde.add_vline(
                x=thresh_val, line_dash='dash', line_color='#dc2626', line_width=2,
                annotation_text="임계값 {:.4f}".format(thresh_val),
                annotation_font=dict(color='#dc2626', size=10),
            )
            fig_kde.update_layout(
                **PLOT_BASE,
                title="복원 오차 KDE 분포 (임계값 기준 분리)",
                xaxis_title="Reconstruction Error (MSE)",
                yaxis_title="Density",
                height=300,
                legend=dict(x=0.65, y=0.95, font_size=10),
            )
            st.plotly_chart(fig_kde, use_container_width=True, config={"displayModeBar": False})

            c1, c2, c3 = st.columns(3)
            c1.metric("정상 평균 오차", "{:.4f}".format(normal_errors.mean()))
            c2.metric(
                "위협 평균 오차",
                "{:.4f}".format(threat_errors.mean()) if len(threat_errors) > 0 else "N/A",
            )
            ratio = (threat_errors.mean() / max(normal_errors.mean(), 1e-9)) if len(threat_errors) > 0 else None
            c3.metric("분리 비율", "{:.1f}x".format(ratio) if ratio else "N/A")

    with tab2:
        col_cm1, col_cm2 = st.columns([6, 4])

        with col_cm1:
            st.markdown("#### 정규화 혼동 행렬")
            labels    = label_encoder.classes_
            row_sums  = conf_matrix.sum(axis=1, keepdims=True)
            cm_norm   = np.where(row_sums > 0, conf_matrix.astype(float) / row_sums, 0.0)
            fig_cm = px.imshow(
                cm_norm,
                labels=dict(x="예측 레이블", y="실제 레이블", color="비율"),
                x=labels, y=labels,
                color_continuous_scale=[[0, '#f8fafc'], [0.5, '#a5b4fc'], [1, '#4f46e5']],
                text_auto='.2f', zmin=0, zmax=1,
            )
            for i in range(len(labels)):
                for j in range(len(labels)):
                    fig_cm.add_annotation(
                        x=j, y=i,
                        text="<br><span style='font-size:8px'>({:,}건)</span>".format(conf_matrix[i, j]),
                        showarrow=False,
                        font=dict(size=7, color='#475569'),
                    )
            fig_cm.update_layout(
                **PLOT_BASE,
                title="정규화 혼동 행렬 (비율 + 실제 건수)",
                height=420,
            )
            fig_cm.update_coloraxes(showscale=True, colorbar_title="비율")
            st.plotly_chart(fig_cm, use_container_width=True, config={"displayModeBar": False})

        with col_cm2:
            st.markdown("#### 클래스별 정확도")
            per_class_acc = np.where(
                conf_matrix.sum(axis=1) > 0,
                conf_matrix.diagonal() / conf_matrix.sum(axis=1),
                0.0,
            )
            for cls, acc in zip(labels, per_class_acc):
                color = CLASS_COLORS.get(cls, '#94a3b8')
                bar_html = (
                    "<div style='margin-bottom:0.6rem;'>"
                    "<div style='display:flex;justify-content:space-between;margin-bottom:3px;'>"
                    "<span style='font-size:0.8rem;font-weight:600;color:#374151;'>{cls}</span>"
                    "<span style='font-family:JetBrains Mono,monospace;font-size:0.8rem;color:{c};font-weight:700;'>{acc:.1%}</span>"
                    "</div>"
                    "<div style='background:#f1f5f9;border-radius:4px;height:8px;overflow:hidden;'>"
                    "<div style='background:{c};width:{w:.1f}%;height:100%;border-radius:4px;'></div>"
                    "</div>"
                    "</div>"
                ).format(cls=cls, c=color, acc=acc, w=acc * 100)
                st.markdown(bar_html, unsafe_allow_html=True)

            st.divider()
            overall_acc = (conf_matrix.diagonal().sum() / conf_matrix.sum()
                           if conf_matrix.sum() > 0 else 0.0)
            st.metric("전체 정확도 (Accuracy)", "{:.4f}".format(overall_acc))

    with tab3:
        col_r1, col_r2 = st.columns([6, 4])

        with col_r1:
            st.markdown("#### Precision / Recall / F1-Score")
            report_plot = eval_report.reset_index().rename(columns={'index': 'Class'})
            fig_report  = go.Figure()
            metric_cfg = {
                'precision': ('정밀도',   '#4f46e5'),
                'recall':    ('재현율',   '#7c3aed'),
                'f1-score':  ('F1-Score', '#0ea5e9'),
            }
            for col_key, (name, color) in metric_cfg.items():
                fig_report.add_trace(go.Bar(
                    x=report_plot['Class'],
                    y=report_plot[col_key],
                    name=name,
                    marker_color=color,
                    marker_line_width=0,
                    text=report_plot[col_key].apply(lambda v: "{:.3f}".format(v)),
                    textposition='outside',
                    textfont_size=9,
                ))

            merged_layout = PLOT_BASE.copy()
            merged_layout['yaxis'] = dict(
                gridcolor='#f1f5f9', linecolor='#e2e8f0',
                tickfont=dict(size=9, color='#94a3b8'), range=[0, 1.12],
            )

            fig_report.update_layout(
                **merged_layout,
                title="클래스별 분류 성능 세부 지표",
                barmode='group', height=340,
                legend=dict(orientation='h', y=1.04, x=1, xanchor='right', font_size=10),
            )
            st.plotly_chart(fig_report, use_container_width=True, config={"displayModeBar": False})

            st.markdown("#### 클래스별 종합 성능 레이더")
            radar_classes = [c for c in label_encoder.classes_ if c != 'Benign'][:4]
            fig_radar = go.Figure()
            for cls in radar_classes:
                if cls in eval_report.index:
                    row = eval_report.loc[cls]
                    fig_radar.add_trace(go.Scatterpolar(
                        r=[row['precision'], row['recall'], row['f1-score'], row['precision']],
                        theta=['Precision', 'Recall', 'F1-Score', 'Precision'],
                        fill='toself', name=cls,
                        line=dict(color=CLASS_COLORS.get(cls, '#94a3b8'), width=2),
                    ))
            fig_radar.update_layout(
                polar=dict(
                    radialaxis=dict(visible=True, range=[0, 1], tickfont_size=8),
                    angularaxis=dict(tickfont_size=10),
                ),
                paper_bgcolor='#ffffff',
                plot_bgcolor='#ffffff',
                showlegend=True,
                legend=dict(font_size=10),
                height=300,
                title_text="위협 유형별 3축 성능 레이더",
                title_font=dict(size=13, color='#0f172a'),
            )
            st.plotly_chart(fig_radar, use_container_width=True, config={"displayModeBar": False})

        with col_r2:
            st.markdown("#### Classification Report")
            display_report = eval_report.copy()
            display_report.index.name = '클래스'
            display_report.columns    = ['정밀도', '재현율', 'F1']
            st.dataframe(
                display_report.style.format("{:.4f}").background_gradient(cmap='Blues', vmin=0, vmax=1),
                use_container_width=True, height=300,
            )

            st.divider()
            avg_f1   = eval_report['f1-score'].mean()
            avg_prec = eval_report['precision'].mean()
            avg_rec  = eval_report['recall'].mean()
            st.metric("평균 F1-Score", "{:.4f}".format(avg_f1))
            st.metric("평균 정밀도",    "{:.4f}".format(avg_prec))
            st.metric("평균 재현율",    "{:.4f}".format(avg_rec))

            st.divider()
            st.markdown("#### 모델 구성")
            model_info = [
                ("딥러닝 모델",  "{} Autoencoder".format(selected_mode)),
                ("시퀀스 길이",  "5 Timesteps"),
                ("은닉 차원",   "16 Hidden Units"),
                ("학습 에포크",  "10"),
                ("분류기",      "XGBoost" if XGB_AVAILABLE else "RandomForest"),
                ("추정기 수",   "100 Estimators"),
                ("피처 수",     "{}개".format(len(available))),
            ]
            for k, v in model_info:
                row_html = (
                    "<div style='display:flex;justify-content:space-between;"
                    "padding:0.3rem 0;border-bottom:1px solid #f1f5f9;'>"
                    "<span style='font-size:0.78rem;color:#64748b;'>{k}</span>"
                    "<span style='font-family:JetBrains Mono,monospace;font-size:0.78rem;"
                    "color:#0f172a;font-weight:600;'>{v}</span>"
                    "</div>"
                ).format(k=k, v=v)
                st.markdown(row_html, unsafe_allow_html=True)

    with tab4:
        st.markdown("#### 이진 위협 분류 관점 고급 지표")
        st.caption("'Benign'을 정상, 나머지를 위협으로 보는 이진 분류 기준입니다.")

        y_test_names = label_encoder.inverse_transform(y_test)
        y_pred_names = label_encoder.inverse_transform(y_pred)
        y_test_bin   = (y_test_names != 'Benign').astype(int)
        y_pred_bin   = (y_pred_names != 'Benign').astype(int)

        col_adv1, col_adv2 = st.columns(2)

        roc_auc_val = None
        pr_auc_val  = None

        with col_adv1:
            if y_prob is not None:
                threat_idx  = [i for i, c in enumerate(label_encoder.classes_) if c != 'Benign']
                prob_threat = y_prob[:, threat_idx].sum(axis=1)
                fpr, tpr, _ = roc_curve(y_test_bin, prob_threat)
                roc_auc_val = auc(fpr, tpr)

                fig_roc = go.Figure()
                fig_roc.add_trace(go.Scatter(
                    x=fpr, y=tpr, mode='lines',
                    name="ROC (AUC={:.4f})".format(roc_auc_val),
                    line=dict(color='#6366f1', width=2.5),
                ))
                fig_roc.add_trace(go.Scatter(
                    x=[0, 1], y=[0, 1], mode='lines',
                    line=dict(color='#cbd5e1', dash='dash', width=1.5),
                    showlegend=False,
                ))
                fig_roc.add_annotation(
                    x=0.6, y=0.25,
                    text="<b>AUC = {:.4f}</b>".format(roc_auc_val),
                    showarrow=False,
                    font=dict(size=14, color='#6366f1'),
                    bgcolor='#f0f0ff', borderpad=8,
                )
                fig_roc.update_layout(
                    **PLOT_BASE,
                    title="ROC Curve (이진: 정상 vs 위협)",
                    xaxis_title="False Positive Rate",
                    yaxis_title="True Positive Rate",
                    height=320,
                    legend=dict(x=0.45, y=0.08, font_size=10),
                )
                st.plotly_chart(fig_roc, use_container_width=True, config={"displayModeBar": False})
                st.metric("AUC-ROC", "{:.4f}".format(roc_auc_val),
                          help="1.0에 가까울수록 이상적")
            else:
                st.info("확률값 미지원 모델 — ROC Curve 생략")

        with col_adv2:
            if y_prob is not None:
                prec_curve, rec_curve, _ = precision_recall_curve(y_test_bin, prob_threat)
                pr_auc_val = auc(rec_curve, prec_curve)

                fig_pr = go.Figure()
                fig_pr.add_trace(go.Scatter(
                    x=rec_curve, y=prec_curve, mode='lines',
                    name="PR Curve (AUC={:.4f})".format(pr_auc_val),
                    line=dict(color='#0ea5e9', width=2.5),
                    fill='tozeroy', fillcolor='rgba(14,165,233,0.07)',
                ))
                baseline = y_test_bin.mean()
                fig_pr.add_hline(
                    y=baseline, line_dash='dash',
                    line_color='#cbd5e1', line_width=1.5,
                    annotation_text="Baseline {:.3f}".format(baseline),
                    annotation_font=dict(size=9, color='#94a3b8'),
                )
                fig_pr.update_layout(
                    **PLOT_BASE,
                    title="Precision-Recall Curve",
                    xaxis_title="Recall",
                    yaxis_title="Precision",
                    height=320,
                    legend=dict(x=0.35, y=0.08, font_size=10),
                )
                st.plotly_chart(fig_pr, use_container_width=True, config={"displayModeBar": False})
                st.metric("PR-AUC", "{:.4f}".format(pr_auc_val),
                          help="불균형 데이터에서 ROC보다 신뢰도 높음")
            else:
                st.info("확률값 미지원 모델 — PR Curve 생략")

        st.divider()
        st.markdown("#### 이진 분류 성능 지표 요약")
        bin_cols = st.columns(4)
        bin_metrics = [
            ("정확도 (Accuracy)",     accuracy_score(y_test_bin, y_pred_bin)),
            ("정밀도 (Precision)",     precision_score(y_test_bin, y_pred_bin, zero_division=0)),
            ("재현율 (Recall / TPR)", recall_score(y_test_bin, y_pred_bin, zero_division=0)),
            ("F1-Score",              f1_score(y_test_bin, y_pred_bin, zero_division=0)),
        ]
        for i, (name, val) in enumerate(bin_metrics):
            bin_cols[i].metric(name, "{:.4f}".format(val))

        with st.expander("학술 검증 요약", expanded=True):
            auc_str = "{:.4f}".format(roc_auc_val) if roc_auc_val is not None else "N/A"
            quality = "(우수)" if roc_auc_val and roc_auc_val > 0.95 else "(개선 필요)"
            avg_f1_local = eval_report['f1-score'].mean()
            st.success(f"""{{selected_mode}} Autoencoder + {{clf_name}} 융합 파이프라인

- 딥러닝 복원 오차 기반 비지도 이상 탐지: 임계값 {{thresh_val:.5f}} / 오염률 {{calculated_contam * 100:.2f}}%
- 지도학습 다중 분류 F1: {{avg_f1_local:.4f}} (Macro Avg)
- 이진 위협 탐지 AUC-ROC: {{auc_str}} {{quality}}
- 시계열 시퀀스 길이 5 Timestep 적용으로 단순 통계 모델 대비 장기 의존성 맥락 보존""")
''')

with open('app.py', 'w', encoding='utf-8') as f:
    f.write('\n'.join(app_lines))

print("app.py 작성 완료.")

streamlit_bin = shutil.which('streamlit') or os.path.join(os.path.dirname(sys.executable), 'streamlit')

subprocess.Popen(
    [streamlit_bin, 'run', 'app.py',
     '--server.headless=true',
     '--server.port=8501',
     '--server.fileWatcherType=none'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(5)

print("Cloudflared 터널 연동 시작...")
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
os.system('chmod +x cloudflared')

with open('tunnel.log', 'w') as log_file:
    subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8501'],
        stdout=log_file, stderr=log_file,
    )

print("터널 도메인 주소 할당 대기 중...")
url_found = False

for attempt in range(5):
    time.sleep(3)
    if os.path.exists('tunnel.log'):
        with open('tunnel.log', 'r') as log_file:
            log_content = log_file.read()
            if 'trycloudflare.com' in log_content:
                for line in log_content.split('\n'):
                    if 'trycloudflare.com' in line:
                        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                        if m:
                            print('=' * 52)
                            print('접속 URL: ' + m.group(0))
                            print('=' * 52)
                            url_found = True
                            break
            if url_found:
                break

if not url_found:
    print('=' * 52)
    print('URL 자동 추출 실패 → !cat tunnel.log | grep trycloudflare.com')
    print('=' * 52)